In [1]:
# ==============================================================================
# K-FOLD CROSS-VALIDATION CHO TABNET — ĐẦY ĐỦ CHỈ SỐ
# (Accuracy, Macro/Weighted Precision-Recall-F1, Macro AUC, per-class chi tiết)
# ==============================================================================
# CẢNH BÁO VỀ THỜI GIAN (đọc trước khi chạy):
#   - K-fold = train lại TabNet TỪ ĐẦU N_SPLITS lần → tốn gấp N_SPLITS lần
#     thời gian so với 1 lần train hiện có.
#   - Để khả thi trên CPU, code này GIẢM max_epochs (100→40) và patience
#     (15→8) so với cấu hình gốc. Đây là đánh đổi cần NÊU RÕ trong luận văn.
#   - Nếu vẫn quá lâu, đổi N_SPLITS = 3 thay vì 5.
#   - Kết quả TẠM được lưu sau mỗi fold — nếu phải Interrupt giữa chừng,
#     không mất hết các fold đã chạy xong.
# ==============================================================================

import time
import json
import pickle
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

# --- 1. Tải dữ liệu ---
data_dir = "../data/processed/"

X_train = pd.read_csv(data_dir + "X_train.csv")
X_val   = pd.read_csv(data_dir + "X_val.csv")
X_test  = pd.read_csv(data_dir + "X_test.csv")

y_train = pd.read_csv(data_dir + "y_train.csv").values.ravel()
y_val   = pd.read_csv(data_dir + "y_val.csv").values.ravel()
y_test  = pd.read_csv(data_dir + "y_test.csv").values.ravel()

with open(data_dir + "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
label_names = label_encoder.classes_
n_classes = len(label_names)

# --- 2. Gộp lại toàn bộ dữ liệu để K-fold tự chia lại ---
X_all = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)
y_all = np.concatenate([y_train, y_val, y_test])
X_all_np = X_all.values

print(f"Tổng số mẫu sau khi gộp: {X_all_np.shape}")
print(f"Phân bố lớp trên toàn bộ dữ liệu:")
for idx, name in enumerate(label_names):
    cnt = np.sum(y_all == idx)
    print(f"  {name:<25} {cnt:>8} mẫu ({cnt/len(y_all)*100:.2f}%)")


# --- 3. Cấu hình TabNet — GIỮ NGUYÊN kiến trúc gốc, chỉ giảm epoch/patience ---
TABNET_PARAMS = {
    "n_d": 64, "n_a": 64, "n_steps": 5, "gamma": 1.5,
    "n_independent": 2, "n_shared": 2, "lambda_sparse": 1e-4,
    "momentum": 0.3, "clip_value": 2.0,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": dict(lr=2e-2),
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {"step_size": 10, "gamma": 0.9},
    "mask_type": "entmax",
    "verbose": 0,
    "seed": 42,
}

N_SPLITS = 5          # đổi thành 3 nếu quá lâu
MAX_EPOCHS = 50        # giảm so với 100 gốc
PATIENCE = 8            # giảm so với 15 gốc
BATCH_SIZE = 1024
VIRTUAL_BATCH_SIZE = 128


# --- 4. Chạy Stratified K-Fold ---
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_results = []

overall_t0 = time.perf_counter()

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_all_np, y_all), 1):
    print(f"\n{'='*70}")
    print(f"FOLD {fold_idx}/{N_SPLITS}")
    print(f"{'='*70}")

    X_tr_full, X_te = X_all_np[train_idx], X_all_np[test_idx]
    y_tr_full, y_te = y_all[train_idx], y_all[test_idx]

    # Tách thêm 10% từ train để làm validation set (phục vụ early stopping)
    rng = np.random.RandomState(42 + fold_idx)
    perm = rng.permutation(len(X_tr_full))
    val_size = int(0.1 * len(X_tr_full))
    val_idx_local, tr_idx_local = perm[:val_size], perm[val_size:]

    X_tr, X_va = X_tr_full[tr_idx_local], X_tr_full[val_idx_local]
    y_tr, y_va = y_tr_full[tr_idx_local], y_tr_full[val_idx_local]

    print(f"  Train: {X_tr.shape[0]} mẫu | Val (early stop): {X_va.shape[0]} mẫu | "
          f"Test (fold): {X_te.shape[0]} mẫu")

    model_fold = TabNetClassifier(**TABNET_PARAMS)

    t0 = time.perf_counter()
    model_fold.fit(
        X_train=X_tr, y_train=y_tr,
        eval_set=[(X_va, y_va)],
        eval_name=["valid"],
        eval_metric=["balanced_accuracy"],
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        batch_size=BATCH_SIZE,
        virtual_batch_size=VIRTUAL_BATCH_SIZE,
        num_workers=0,
        drop_last=False
    )
    train_time = time.perf_counter() - t0
    n_epochs_actual = len(model_fold.history.history['loss'])

    y_pred = model_fold.predict(X_te)
    y_proba = model_fold.predict_proba(X_te)

    # --- Chỉ số tổng quát ---
    acc = accuracy_score(y_te, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_te, y_pred, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_te, y_pred, average='weighted', zero_division=0
    )
    try:
        y_te_bin = label_binarize(y_te, classes=range(n_classes))
        macro_auc = roc_auc_score(y_te_bin, y_proba, multi_class='ovr', average='macro')
    except Exception as e:
        print(f"  ⚠ Không tính được AUC cho fold {fold_idx}: {e}")
        macro_auc = None

    # --- Chỉ số per-class đầy đủ ---
    report = classification_report(
        y_te, y_pred, target_names=label_names,
        output_dict=True, zero_division=0
    )
    per_class_precision = {cls: report[cls]['precision'] for cls in label_names}
    per_class_recall = {cls: report[cls]['recall'] for cls in label_names}
    per_class_f1 = {cls: report[cls]['f1-score'] for cls in label_names}
    per_class_support = {cls: int(report[cls]['support']) for cls in label_names}

    fold_results.append({
        'fold': fold_idx,
        'train_time_sec': round(train_time, 1),
        'n_epochs_actual': n_epochs_actual,
        'accuracy': acc,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'weighted_precision': weighted_p,
        'weighted_recall': weighted_r,
        'weighted_f1': weighted_f1,
        'macro_auc': macro_auc,
        'per_class_precision': per_class_precision,
        'per_class_recall': per_class_recall,
        'per_class_f1': per_class_f1,
        'per_class_support': per_class_support,
    })

    elapsed_total = time.perf_counter() - overall_t0
    auc_str = f"{macro_auc:.4f}" if macro_auc is not None else "N/A"
    print(f"  Accuracy={acc:.4f}  Macro F1={macro_f1:.4f}  Macro AUC={auc_str}  "
          f"Weighted F1={weighted_f1:.4f}  ({n_epochs_actual} epochs, {train_time:.1f}s)")
    print(f"  mitmattack:  P={per_class_precision['mitmattack']:.4f}  "
          f"R={per_class_recall['mitmattack']:.4f}  F1={per_class_f1['mitmattack']:.4f}  "
          f"(support={per_class_support['mitmattack']})")
    print(f"  floodattack: P={per_class_precision['floodattack']:.4f}  "
          f"R={per_class_recall['floodattack']:.4f}  F1={per_class_f1['floodattack']:.4f}  "
          f"(support={per_class_support['floodattack']})")
    print(f"  [Tổng thời gian đã chạy: {elapsed_total/60:.1f} phút]")

    # Lưu kết quả TẠM sau mỗi fold — phòng trường hợp bị ngắt giữa chừng
    with open(data_dir + 'kfold_tabnet_results_partial.json', 'w', encoding='utf-8') as f:
        json.dump({'n_splits': N_SPLITS, 'fold_results': fold_results}, f,
                   indent=4, ensure_ascii=False)


# --- 5. Tổng hợp mean ± std qua N_SPLITS fold ---
print(f"\n{'='*70}")
print(f"TỔNG HỢP {N_SPLITS}-FOLD — TABNET")
print(f"{'='*70}")

total_time_min = (time.perf_counter() - overall_t0) / 60
print(f"\nTổng thời gian chạy toàn bộ K-fold: {total_time_min:.1f} phút")

accs = [r['accuracy'] for r in fold_results]
mf1s = [r['macro_f1'] for r in fold_results]
mps = [r['macro_precision'] for r in fold_results]
mrs = [r['macro_recall'] for r in fold_results]
wf1s = [r['weighted_f1'] for r in fold_results]
aucs = [r['macro_auc'] for r in fold_results if r['macro_auc'] is not None]

print(f"Accuracy:        {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Macro Precision: {np.mean(mps):.4f} ± {np.std(mps):.4f}")
print(f"Macro Recall:    {np.mean(mrs):.4f} ± {np.std(mrs):.4f}")
print(f"Macro F1:        {np.mean(mf1s):.4f} ± {np.std(mf1s):.4f}")
print(f"Weighted F1:     {np.mean(wf1s):.4f} ± {np.std(wf1s):.4f}")
if aucs:
    print(f"Macro AUC:       {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

print(f"\nPer-class (mean ± std qua {N_SPLITS} fold):")
summary_per_class = {}
for cls_name in label_names:
    cls_p = [r['per_class_precision'][cls_name] for r in fold_results]
    cls_r = [r['per_class_recall'][cls_name] for r in fold_results]
    cls_f1 = [r['per_class_f1'][cls_name] for r in fold_results]
    summary_per_class[cls_name] = {
        'precision_mean': np.mean(cls_p), 'precision_std': np.std(cls_p),
        'recall_mean': np.mean(cls_r), 'recall_std': np.std(cls_r),
        'f1_mean': np.mean(cls_f1), 'f1_std': np.std(cls_f1),
        'fold_f1_values': cls_f1,
    }
    marker = "⚠ " if cls_name in ('mitmattack', 'floodattack') else "  "
    print(f"{marker}{cls_name:<25} "
          f"P={np.mean(cls_p):.4f}±{np.std(cls_p):.4f}  "
          f"R={np.mean(cls_r):.4f}±{np.std(cls_r):.4f}  "
          f"F1={np.mean(cls_f1):.4f}±{np.std(cls_f1):.4f}")


# --- 6. Lưu kết quả cuối cùng ---
output = {
    'n_splits': N_SPLITS,
    'model': 'TabNet',
    'max_epochs_used': MAX_EPOCHS,
    'patience_used': PATIENCE,
    'note': 'max_epochs/patience giảm so với cấu hình gốc (100/15) để khả thi về thời gian trên CPU',
    'total_runtime_min': round(total_time_min, 1),
    'fold_results': fold_results,
    'summary': {
        'accuracy_mean': float(np.mean(accs)), 'accuracy_std': float(np.std(accs)),
        'macro_precision_mean': float(np.mean(mps)), 'macro_precision_std': float(np.std(mps)),
        'macro_recall_mean': float(np.mean(mrs)), 'macro_recall_std': float(np.std(mrs)),
        'macro_f1_mean': float(np.mean(mf1s)), 'macro_f1_std': float(np.std(mf1s)),
        'weighted_f1_mean': float(np.mean(wf1s)), 'weighted_f1_std': float(np.std(wf1s)),
        'macro_auc_mean': float(np.mean(aucs)) if aucs else None,
        'macro_auc_std': float(np.std(aucs)) if aucs else None,
        'per_class_summary': {
            cls: {k: (float(v) if not isinstance(v, list) else v) for k, v in stats.items()}
            for cls, stats in summary_per_class.items()
        }
    }
}

with open(data_dir + 'kfold_tabnet_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"\n✓ Đã lưu: {data_dir}kfold_tabnet_results.json")
print("\nSo sánh nhanh với kết quả 1-lần-test trước đó:")
print(f"  1 lần test (gốc):  mitmattack F1=0.4444,  floodattack F1=0.8387")
print(f"  5-fold mean:       mitmattack F1={summary_per_class['mitmattack']['f1_mean']:.4f} ± "
      f"{summary_per_class['mitmattack']['f1_std']:.4f},  "
      f"floodattack F1={summary_per_class['floodattack']['f1_mean']:.4f} ± "
      f"{summary_per_class['floodattack']['f1_std']:.4f}")

print("\nSo với kfold_random_forest_results.json (nếu đã chạy trước) để đối chiếu"
      " độ ổn định giữa 2 mô hình.")

Tổng số mẫu sau khi gộp: (319971, 78)
Phân bố lớp trên toàn bộ dữ liệu:
  attackfree                     255 mẫu (0.08%)
  dosattack                   304627 mẫu (95.20%)
  floodattack                    108 mẫu (0.03%)
  fuzzyattack                    939 mẫu (0.29%)
  iec104starvationattack        2028 mẫu (0.63%)
  mitmattack                      26 mẫu (0.01%)
  ntpddosattack                 2278 mẫu (0.71%)
  portscanattack                9710 mẫu (3.03%)

FOLD 1/5
  Train: 230379 mẫu | Val (early stop): 25597 mẫu | Test (fold): 63995 mẫu

Early stopping occurred at epoch 15 with best_epoch = 7 and best_valid_balanced_accuracy = 0.91433
  Accuracy=0.9960  Macro F1=0.8651  Macro AUC=0.9959  Weighted F1=0.9959  (16 epochs, 1625.7s)
  mitmattack:  P=0.3333  R=0.8000  F1=0.4706  (support=5)
  floodattack: P=1.0000  R=0.5909  F1=0.7429  (support=22)
  [Tổng thời gian đã chạy: 27.4 phút]

FOLD 2/5
  Train: 230380 mẫu | Val (early stop): 25597 mẫu | Test (fold): 63994 mẫu

Early stopping